<a href="https://colab.research.google.com/github/Shyvanos/bxmid03-hs26-team-3/blob/main/checks/Team3_Ortbeton_NPK_Starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="In Colab öffnen"/></a>

# Team 3 · Ortbetonwände → NPK-Positionen (Starter)

Dieses Notebook ist ein **Gerüst**. Es liest die Wände aus dem IFC, macht den Scope sichtbar, zeigt Datenlücken und rechnet Summen pro Position.

**Was ihr selbst entscheidet und begründet:**
1. den Scope (Schritt 4),
2. die Zuordnung Betontyp × Höhe → NPK-Position (Schritt 6). Die Tabelle enthält nur **eine Beispielzeile**, die ihr ersetzt.

Ausgabe: `ortbeton_pro_wand.csv` (eine Zeile pro Wand: GlobalId, Ist-Werte, Position, Status, Grund) und `ortbeton_summen.csv` (m³ pro Position).

Werkzeuge: Python 3.12, IfcOpenShell 0.9.0.post1, pandas.

## 1 · Pakete installieren
In Colab nötig, lokal nur einmal.

In [ ]:
%pip install -q ifcopenshell==0.9.0.post1 pandas

## 2 · Modell laden

Das Arbeitsmodell ist **geschützt** und gehört nicht ins Repository.
- **Lokal (bevorzugt):** Notebook in VS Code oder Jupyter öffnen und `MODELL` auf euren lokalen Pfad setzen.
- **Colab:** Nur wenn das für euer Modell erlaubt ist. Dann fragt die Zelle nach der Datei (Upload auf Google-Server).

In [ ]:
import os
MODELL = "Team_3_WaltGalmarini_Arbeitsmodell.ifc"   # Pfad anpassen
MODELL = os.environ.get("TEAM3_MODELL", MODELL)

if not os.path.exists(MODELL):
    try:
        from google.colab import files
        MODELL = next(iter(files.upload()))
    except ImportError:
        raise FileNotFoundError(f"Modell nicht gefunden: {MODELL}. Pfad oben anpassen.")

import ifcopenshell, ifcopenshell.util.element as ue, ifcopenshell.util.unit as uu
modell = ifcopenshell.open(MODELL)
skala = uu.calculate_unit_scale(modell)   # Längeneinheit → Meter
print(f"{MODELL}: {modell.schema}, Längeneinheit × {skala} = m")
print({k: len(modell.by_type(k)) for k in ("IfcWall", "IfcSlab", "IfcColumn")})

## 3 · Wandtabelle
Eine Zeile pro `IfcWall`. Fehlende Werte bleiben **leer** (NaN), nicht 0.

In [ ]:
import pandas as pd

def wert(psets, pset, name):
    return psets.get(pset, {}).get(name)

zeilen = []
for w in modell.by_type("IfcWall"):
    p = ue.get_psets(w)
    mat = ue.get_material(w)
    hoehe = wert(p, "Qto_WallBaseQuantities", "Height")
    zeilen.append({
        "GlobalId": w.GlobalId,
        "Name": w.Name,
        "Betontyp": wert(p, "wg_SUB", "Betontyp"),
        "Ausfuehrung": wert(p, "wg_SUB", "Ausfuehrung"),
        "Material": getattr(mat, "Name", None),
        "eBKP_H": wert(p, "wg_SUB", "eBKP-H"),
        "Hoehe_m": hoehe * skala if hoehe is not None else None,
        "Dicke_m": (wert(p, "Qto_WallBaseQuantities", "Width") or 0) * skala or None,
        "NetVolume_m3": wert(p, "Qto_WallBaseQuantities", "NetVolume"),
        "NetSideArea_m2": wert(p, "Qto_WallBaseQuantities", "NetSideArea"),
        "Betonieretappe": wert(p, "CH_Ortbeton", "Betonieretappe"),
        "Schalungstyp": wert(p, "CH_Ortbeton", "Schalungstyp"),
    })
df = pd.DataFrame(zeilen)
print(len(df), "Wände")
df.head()

Was steckt drin? Häufigkeiten der Felder, die ihr für Scope und Zuordnung braucht:

In [ ]:
for spalte in ("Ausfuehrung", "Betontyp", "Material", "Betonieretappe"):
    print(df[spalte].value_counts(dropna=False).to_string(), "\n")

## 4 · Scope: welche Wände gehören in den Mengenauszug?

**Entscheid des Teams.** Unten steht ein Vorschlag als Startpunkt. Prüft ihn und begründet jede Regel im README.
Jede Wand bekommt `im_Scope` (ja/nein) und einen Grund.

In [ ]:
def scope(z):
    # Vorschlag: anpassen und begründen
    if z.Ausfuehrung != "Ortbeton":
        return False, f"Ausführung '{z.Ausfuehrung}' ist nicht Ortbeton"
    if isinstance(z.Material, str) and "Bestand" in z.Material:
        return False, "Bestand, kein neues Bauteil"
    return True, ""

df[["im_Scope", "Grund_Scope"]] = df.apply(lambda z: pd.Series(scope(z)), axis=1)
print(df["im_Scope"].value_counts().to_string())
df.loc[~df["im_Scope"], ["GlobalId", "Betontyp", "Material", "Grund_Scope"]]

## 5 · Datenprüfung im Scope
Nur zählen und auflisten, nichts reparieren. Jeder Fund ist ein Befund für den Modellautor.

In [ ]:
s = df[df["im_Scope"]]
pruefungen = {
    "Betontyp fehlt": s["Betontyp"].isna(),
    "Höhe fehlt": s["Hoehe_m"].isna(),
    "NetVolume fehlt oder ≤ 0": ~(s["NetVolume_m3"] > 0),
    "Höhe über 4.0 m": s["Hoehe_m"] > 4.0,
}
for name, maske in pruefungen.items():
    print(f"{name}: {int(maske.sum())} von {len(s)}")

In [ ]:
# Beispiel: welche Wände haben keine Höhe?
s.loc[s["Hoehe_m"].isna(), ["GlobalId", "Name", "Betontyp", "NetVolume_m3"]]

## 6 · Zuordnung Betontyp × Höhe → NPK-Position

**Hier liegt eure Fachleistung.** Pro Zeile: welche Wände (Betontyp und Höhenbereich) gehören zu welcher Position, in welcher Einheit, und **warum**.
- Quelle für Positionen: euer NPK 241 (CRB-Lizenz, Positionstexte nicht ins Repository kopieren, nur Nummer und eigene Kurzbezeichnung).
- Was bedeuten die Betontypen `01`, `02`, `03` im Modell? Quelle angeben (Modellautor, Legende, eigene Annahme).
- Bereiche dürfen sich nicht überschneiden. Wände ausserhalb aller Bereiche bleiben «nicht zugeordnet».

Die einzige Zeile ist ein **Beispiel** und muss ersetzt werden.

In [ ]:
ZUORDNUNG = pd.DataFrame([
    # Betontyp, Höhe von (m, inkl.), Höhe bis (m, exkl.), Position, Einheit, Begründung
    {"Betontyp": "03", "Hoehe_von_m": 0.0, "Hoehe_bis_m": 99.0,
     "NPK_Position": "BEISPIEL", "Einheit": "m3",
     "Begruendung": "BEISPIEL: ersetzen durch eigene Position und Begründung"},
])
ZUORDNUNG

## 7 · Status pro Wand
- **zugeordnet:** genau eine Zeile der Zuordnung passt
- **mehrdeutig:** mehrere Zeilen passen (Zuordnung korrigieren)
- **nicht zugeordnet:** keine Zeile passt
- **fehlend:** Betontyp oder Höhe fehlt, Zuordnung nicht möglich
- **ausser Scope:** siehe Schritt 4

In [ ]:
def zuordnen(z):
    if not z.im_Scope:
        return pd.Series({"NPK_Position": None, "Status": "ausser Scope", "Grund": z.Grund_Scope})
    if pd.isna(z.Betontyp) or pd.isna(z.Hoehe_m):
        fehlt = "Betontyp" if pd.isna(z.Betontyp) else "Höhe"
        return pd.Series({"NPK_Position": None, "Status": "fehlend", "Grund": f"{fehlt} fehlt"})
    treffer = ZUORDNUNG[(ZUORDNUNG["Betontyp"] == str(z.Betontyp))
                        & (ZUORDNUNG["Hoehe_von_m"] <= z.Hoehe_m) & (z.Hoehe_m < ZUORDNUNG["Hoehe_bis_m"])]
    if len(treffer) == 1:
        return pd.Series({"NPK_Position": treffer.iloc[0]["NPK_Position"], "Status": "zugeordnet", "Grund": ""})
    if len(treffer) > 1:
        return pd.Series({"NPK_Position": None, "Status": "mehrdeutig",
                          "Grund": "passt zu " + ", ".join(treffer["NPK_Position"])})
    return pd.Series({"NPK_Position": None, "Status": "nicht zugeordnet",
                      "Grund": f"keine Zeile für Betontyp {z.Betontyp}, Höhe {z.Hoehe_m:.2f} m"})

df[["NPK_Position", "Status", "Grund"]] = df.apply(zuordnen, axis=1)
print(df["Status"].value_counts().to_string())

## 8 · Summen pro Position und Kontrolle

In [ ]:
summen = (df[df["Status"] == "zugeordnet"]
          .groupby("NPK_Position")
          .agg(Anzahl_Waende=("GlobalId", "count"), NetVolume_m3=("NetVolume_m3", "sum"))
          .round(2).reset_index())
summen

In [ ]:
# Kontrolle: Summe pro Status = Gesamtvolumen aller Wände, nichts geht verloren
pro_status = df.groupby("Status")["NetVolume_m3"].sum()
print(pro_status.round(2).to_string())
total = df["NetVolume_m3"].sum()
assert abs(pro_status.sum() - total) < 1e-6, "Volumen ging verloren"
print(f"Total {total:.2f} m³ = Summe der Status ✔")

## 9 · Export

In [ ]:
spalten = ["GlobalId", "Name", "Betontyp", "Hoehe_m", "NetVolume_m3", "NPK_Position", "Status", "Grund"]
df[spalten].to_csv("ortbeton_pro_wand.csv", sep=";", index=False, encoding="utf-8-sig")
summen.to_csv("ortbeton_summen.csv", sep=";", index=False, encoding="utf-8-sig")
df.loc[df["Status"].isin(["fehlend", "nicht zugeordnet", "mehrdeutig"]), spalten] \
  .to_csv("ortbeton_befunde.csv", sep=";", index=False, encoding="utf-8-sig")
print("geschrieben: ortbeton_pro_wand.csv, ortbeton_summen.csv, ortbeton_befunde.csv")
try:
    from google.colab import files
    for f in ("ortbeton_pro_wand.csv", "ortbeton_summen.csv", "ortbeton_befunde.csv"):
        files.download(f)
except ImportError:
    pass

## 10 · Fragen für euer MVP
- **Scope:** Gehört die Wand ohne Betontyp dazu? Was ist mit Bestand?
- **Höhe:** Was macht ihr mit den Wänden ohne Höhe? Was verlangt die NPK bei Wänden über 4 m?
- **Menge:** Ist `NetVolume` die Menge, die die Position verlangt (Messregel, Öffnungsabzug)? Braucht ihr für die Schalung `NetSideArea`?
- **Etappen:** Wollt ihr die Summen zusätzlich nach `Betonieretappe` aufteilen?
- **Nachweis:** Eine Wand von Hand nachrechnen (Volumen, Position) und im README zeigen. Negativfall: Testkopie mit einer Wand ohne Betontyp → Status «fehlend».